In [6]:
import pandas as pd
from sqlalchemy import create_engine, text

# 1. 파일 경로 설정 (내 맥북 로컬 경로)
parquet_path = "/Users/ejpark/workspace/kind/volume/shared/2026w09.zstd.parquet"

# 2. Parquet 파일 로드 및 데이터 확인
df = pd.read_parquet(parquet_path)
print(f"Total Rows: {len(df)}")
display(df.head()) # 데이터 구조 미리보기

Total Rows: 9385578


,app_id_hash,country,city,device_board,device_brand,device_carrier_code,device_carrier_name,device_device,device_display_hardware,device_display_height,...,device_iid_hash,device_language,device_locale,device_manufacturer,device_model,device_product,device_sim_code,device_tac,device_total_memory,timestamp
0,bBuoNuXD20vs46Tkej9mvQ==,US,Grand Rapids,G92BoardV1,8849,310260,Tello,TANK2PRO,TANK2PRO_20240411,2460.0,...,Sg8SWTAGriFRTRlntz1PKw==,en,US,Oblue,TANK 2 PRO,TANK2pro,310240,35074458,1.231503e+10,2026-02-23 03:03:39+00:00
1,AXDll7wYh42TsK8Kz6+iJw==,US,Mays Landing,crosshatch,google,310260,Google Fi,crosshatch,SP1A.210812.016.C2,2960.0,...,Th52ttAv/VxE0rPlW0gguQ==,en,US,Google,Pixel 3 XL,crosshatch,310260,99001201,3.753300e+09,2026-02-23 06:37:38+00:00
2,AXDll7wYh42TsK8Kz6+iJw==,US,North Augusta,tokay,google,310260,Google Fi,tokay,BP4A.260205.002,2424.0,...,1R+8y9uKFZGOabNJs3OF0g==,en,US,Google,Pixel 9,tokay,310240,35562419,1.213340e+10,2026-02-23 07:16:00+00:00
3,bBuoNuXD20vs46Tkej9mvQ==,US,Philadelphia,bluejay,google,311480,XFINITY Mobile,bluejay,BP4A.251205.006,2400.0,...,+WT/avXh7qtNARLGNVnAIA==,en,US,Google,Pixel 6a,bluejay,311480,35016289,5.860991e+09,2026-02-23 10:24:43+00:00
4,AXDll7wYh42TsK8Kz6+iJw==,US,Lexington,hiphi,motorola,311480,Visible,hiphi,U1SHS34.1-177-7-3,2400.0,...,UgmqqMQBGvlsZFX6JHvxgw==,en,US,Motorola,motorola edge plus (2022),hiphi_gu,311480,35643969,7.638450e+09,2026-02-23 09:37:33+00:00


In [7]:
# 3. StarRocks 연결 (MySQL 호환 프로토콜 활용)
SR_USER = "root"
SR_PASS = ""
SR_HOST = "172.16.1.65" 
SR_PORT = "9030" # FE Query Port
SR_DB = "huq"

# 연결 엔진 생성
connection_str = f"mysql+pymysql://{SR_USER}:{SR_PASS}@{SR_HOST}:{SR_PORT}/{SR_DB}"
engine = create_engine(connection_str)

In [ ]:
first_col = df.columns[0]
columns_def = ", ".join([f"`{col}` {('VARCHAR(65533)' if str(df[col].dtype) == 'object' else 'BIGINT' if 'int' in str(df[col].dtype) else 'BIGINT')}" for col in df.columns])

create_table_sql = """
CREATE TABLE bronze (
    -- Key 컬럼들을 맨 앞으로 배치
    app_id_hash             VARCHAR(255),
    `timestamp`             DATETIME,
    
    -- 나머지 일반 컬럼들
    country                 VARCHAR(10),
    city                    VARCHAR(100),
    device_board            VARCHAR(100),
    device_brand            VARCHAR(100),
    device_carrier_code     VARCHAR(50),
    device_carrier_name     VARCHAR(100),
    device_device           VARCHAR(100),
    device_display_hardware VARCHAR(255),
    device_display_height   VARCHAR(50),
    device_display_width    VARCHAR(50),
    device_fingerprint      VARCHAR(500),
    device_hardware         VARCHAR(100),
    device_host             VARCHAR(100),
    device_iid_hash         VARCHAR(255),
    device_language         VARCHAR(10),
    device_locale           VARCHAR(10),
    device_manufacturer     VARCHAR(100),
    device_model            VARCHAR(100),
    device_product          VARCHAR(100),
    device_sim_code         VARCHAR(50),
    device_tac              VARCHAR(50),
    device_total_memory     VARCHAR(50)
) ENGINE=OLAP
DUPLICATE KEY(app_id_hash, `timestamp`)
DISTRIBUTED BY HASH(app_id_hash) BUCKETS 16;
"""

In [ ]:
with engine.begin() as conn:
    # 기존 테이블이 있다면 삭제 (if_exists='replace' 효과)
    conn.execute(text("DROP TABLE IF EXISTS bronze"))
    
    # 새 테이블 생성
    conn.execute(text(create_table_sql))
    print("✅ 테이블 'bronze' 생성 완료")

In [ ]:
cols = [
'app_id_hash', 'timestamp', 'country', 'city', 'device_board', 'device_brand', 
'device_carrier_code', 'device_carrier_name', 'device_device', 'device_display_hardware', 
'device_display_height', 'device_display_width', 'device_fingerprint', 'device_hardware', 
'device_host', 'device_iid_hash', 'device_language', 'device_locale', 'device_manufacturer', 
'device_model', 'device_product', 'device_sim_code', 'device_tac', 'device_total_memory'
]

In [23]:
try:
    df[cols].to_sql(name='bronze', con=engine, if_exists='append', index=False)
    print("🚀 데이터 로딩 성공!")
except Exception as e:
    print(f"❌ 로딩 중 에러: {e}")

✅ 테이블 'bronze' 생성 완료
❌ 로딩 중 에러: Can't reconnect until invalid transaction is rolled back.  Please rollback() fully before proceeding (Background on this error at: https://sqlalche.me/e/20/8s2b)
